In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine
import logging

In [2]:
# Ensure 'logs' directory exists
log_directory = "logs"
os.makedirs(log_directory, exist_ok=True)
# Configure logging
logging.basicConfig(
    filename=os.path.join(log_directory, "app.log"),  # Updated filename
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode='a'
)
logging.info("Logging setup complete!")


In [3]:
# Create SQLite database connection
engine = create_engine('sqlite:///inventory.db')

# Function to ingest data into the database in chunks
def ingest_db(df_iterator, table_name, engine):
    for chunk in df_iterator:
        chunk.to_sql(table_name, con=engine, if_exists='append', index=False)

# Process CSV files in the "data" folder
for file in os.listdir('data'):
    if file.endswith('.csv'):  # More precise file checking
        file_path = os.path.join('data', file)
        
        # Read the CSV file in chunks to reduce memory usage
        chunk_size = 100000  # Adjust based on available RAM
        dtype_mapping = {"col1": "int32", "col2": "float32", "col3": "category"}  # Optimize dtype
        
        df_iterator = pd.read_csv(file_path, chunksize=chunk_size, dtype=dtype_mapping)
        # Logging Info
        logging.info(f'Ingesting {file} in db')        
        # Insert data into the database
        ingest_db(df_iterator, file[:-4], engine)
logging.info('Ingetion Complete')
print("Data ingestion completed successfully!")

Data ingestion completed successfully!
